In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_score

In [7]:
df = pd.read_csv('nt_crime_processed.csv')
print(df.columns)
#Inspection

Index(['Unnamed: 0', 'region', 'Year', 'Month number', 'Alcohol involvement',
       'DV involvement', 'Total_population', 'Aboriginal', 'Non-Aboriginal',
       'Male', 'Female', 'Pop_age_0', 'Pop_age_10', 'Pop_age_15', 'Pop_age_20',
       'Pop_age_25', 'Pop_age_30', 'Pop_age_35', 'Pop_age_40', 'Pop_age_45',
       'Pop_age_5', 'Pop_age_50', 'Pop_age_55', 'Pop_age_60', 'Pop_age_65',
       'Pop_age_70', 'Pop_age_75', 'Pop_age_80', 'Pop_age_85plus',
       'Cask Wine PAC', 'Bottled Wine PAC', 'Fortified Wine PAC', 'Cider PAC',
       'Standard Spirits PAC', 'Mixed Spirits PAC', 'Full-Strength Beer PAC',
       'Mid-Strength Beer PAC', 'Low-Strength Beer PAC', 'Total PAC',
       'Number of offences', 'assault_rate', 'sin_month', 'cos_month',
       'pct_aboriginal', 'pct_male', 'pct_youth', 'pct_senior',
       'alcohol_per_capita', 'full-strength_beer_pac_per_capita',
       'mid-strength_beer_pac_per_capita', 'low-strength_beer_pac_per_capita',
       'cask_wine_pac_per_capita', 'bo

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------------------------------------
# 1. LOAD DATA (region × month)
# ---------------------------------------------------------
df = pd.read_csv("nt_crime_processed.csv")

# ---------------------------------------------------------
# 2. AGGREGATE TO REGION × YEAR
# ---------------------------------------------------------
year_df = df.groupby(['region', 'Year']).agg({
    'Total_population': 'mean',
    'Alcohol involvement': 'mean',
    'DV involvement': 'mean',
    'pct_aboriginal': 'mean',
    'pct_male': 'mean',
    'pct_youth': 'mean',
    'pct_senior': 'mean',
    'alcohol_per_capita': 'mean',
    'full-strength_beer_pac_per_capita': 'mean',
    'mid-strength_beer_pac_per_capita': 'mean',
    'low-strength_beer_pac_per_capita': 'mean',
    'cask_wine_pac_per_capita': 'mean',
    'bottled_wine_pac_per_capita': 'mean',
    'fortified_wine_pac_per_capita': 'mean',
    'assault_rate': 'mean'   # used to create risk class
}).reset_index()

# ---------------------------------------------------------
# 3. CREATE YEAR-LEVEL RISK CLASS
# ---------------------------------------------------------
year_df['risk_class'] = pd.qcut(
    year_df['assault_rate'],
    q=3,
    labels=['low', 'medium', 'high']
)

# ---------------------------------------------------------
# 4. SELECT FEATURES (NO LAG FEATURES)
# ---------------------------------------------------------
feature_cols = [
    'Total_population',
    'Alcohol involvement',
    'DV involvement',
    'pct_aboriginal',
    'pct_male',
    'pct_youth',
    'pct_senior',
    'alcohol_per_capita',
    'full-strength_beer_pac_per_capita',
    'mid-strength_beer_pac_per_capita',
    'low-strength_beer_pac_per_capita',
    'cask_wine_pac_per_capita',
    'bottled_wine_pac_per_capita',
    'fortified_wine_pac_per_capita'
]

# ---------------------------------------------------------
# 5. TRAIN ON 2024, TEST ON 2025
# ---------------------------------------------------------
train = year_df[year_df['Year'] == 2024]
test  = year_df[year_df['Year'] == 2025]

X_train = train[feature_cols]
y_train = train['risk_class']

X_test = test[feature_cols]
y_test = test['risk_class']

# ---------------------------------------------------------
# 6. MULTINOMIAL LOGISTIC REGRESSION
# ---------------------------------------------------------
logit_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        max_iter=500
    ))
])

logit_pipe.fit(X_train, y_train)
y_pred_logit = logit_pipe.predict(X_test)

print("\n=== MULTINOMIAL LOGISTIC REGRESSION (2025) ===")
print(confusion_matrix(y_test, y_pred_logit))
print(classification_report(y_test, y_pred_logit))

# ---------------------------------------------------------
# 7. RANDOM FOREST CLASSIFIER
# ---------------------------------------------------------
rf = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\n=== RANDOM FOREST (2025) ===")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# ---------------------------------------------------------
# 8. RANDOM FOREST FEATURE IMPORTANCE
# ---------------------------------------------------------
rf_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nRANDOM FOREST FEATURE IMPORTANCE")
print(rf_importance)



c:\conda_envs\ai\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



=== MULTINOMIAL LOGISTIC REGRESSION (2025) ===
[[2 0 0]
 [0 2 0]
 [0 0 1]]
              precision    recall  f1-score   support

        high       1.00      1.00      1.00         2
         low       1.00      1.00      1.00         2
      medium       1.00      1.00      1.00         1

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5


=== RANDOM FOREST (2025) ===
[[2 0 0]
 [0 2 0]
 [0 0 1]]
              precision    recall  f1-score   support

        high       1.00      1.00      1.00         2
         low       1.00      1.00      1.00         2
      medium       1.00      1.00      1.00         1

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5


RANDOM FOREST FEATURE IMPORTANCE
                              feature  importance
1                 Al

Research question 5

In [25]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------------------------------------
# 1. LOAD DATA (region × month)
# ---------------------------------------------------------
df = pd.read_csv("nt_crime_processed.csv")

# ---------------------------------------------------------
# 2. CREATE MONTHLY RISK CLASS
# ---------------------------------------------------------
df['risk_class'] = pd.qcut(
    df['assault_rate'],
    q=3,
    labels=['low', 'medium', 'high']
)


# ---------------------------------------------------------
# 4. SELECT FEATURES
# ---------------------------------------------------------
feature_cols = [
    'Total_population',
    'Alcohol involvement',
    'DV involvement',
    'pct_aboriginal',
    'pct_male',
    'pct_youth',
    'pct_senior',
    'alcohol_per_capita',
    'full-strength_beer_pac_per_capita',
    'mid-strength_beer_pac_per_capita',
    'low-strength_beer_pac_per_capita',
    'cask_wine_pac_per_capita',
    'bottled_wine_pac_per_capita',
    'fortified_wine_pac_per_capita',
    'sin_month',
    'cos_month'
]

# ---------------------------------------------------------
# 5. TRAIN ON 2024, TEST ON 2025
# ---------------------------------------------------------
train = df[df['Year'] == 2024]
test  = df[df['Year'] == 2025]

X_train = train[feature_cols]
y_train = train['risk_class']

X_test = test[feature_cols]
y_test = test['risk_class']

# ---------------------------------------------------------
# 6. MULTINOMIAL LOGISTIC REGRESSION
# ---------------------------------------------------------
logit_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        max_iter=500
    ))
])

logit_pipe.fit(X_train, y_train)
y_pred_logit = logit_pipe.predict(X_test)

print("\n=== MULTINOMIAL LOGISTIC REGRESSION (2025) ===")
print(confusion_matrix(y_test, y_pred_logit))
print(classification_report(y_test, y_pred_logit))

# ---------------------------------------------------------
# 7. RANDOM FOREST CLASSIFIER
# ---------------------------------------------------------
rf = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\n=== RANDOM FOREST (2025) ===")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# ---------------------------------------------------------
# 8. RANDOM FOREST FEATURE IMPORTANCE
# ---------------------------------------------------------
rf_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nRANDOM FOREST FEATURE IMPORTANCE")
print(rf_importance)


c:\conda_envs\ai\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



=== MULTINOMIAL LOGISTIC REGRESSION (2025) ===
[[15  1  2]
 [ 0 13  5]
 [ 9  6  9]]
              precision    recall  f1-score   support

        high       0.62      0.83      0.71        18
         low       0.65      0.72      0.68        18
      medium       0.56      0.38      0.45        24

    accuracy                           0.62        60
   macro avg       0.61      0.64      0.62        60
weighted avg       0.61      0.62      0.60        60


=== RANDOM FOREST (2025) ===
[[15  1  2]
 [ 0 15  3]
 [ 9  6  9]]
              precision    recall  f1-score   support

        high       0.62      0.83      0.71        18
         low       0.68      0.83      0.75        18
      medium       0.64      0.38      0.47        24

    accuracy                           0.65        60
   macro avg       0.65      0.68      0.65        60
weighted avg       0.65      0.65      0.63        60


RANDOM FOREST FEATURE IMPORTANCE
                              feature  importance
5 

In [8]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1.Load Data
# ---------------------------------------------------------
df = pd.read_csv("nt_crime_processed.csv")

# ---------------------------------------------------------
# 2. Prepare X and y
# ---------------------------------------------------------
feature_cols = [
    'Total_population',
    'Alcohol involvement',
    'DV involvement',
    'pct_aboriginal',
    'pct_male',
    'pct_youth',
    'pct_senior',
    'alcohol_per_capita',
    'full-strength_beer_pac_per_capita',
    'mid-strength_beer_pac_per_capita',
    'low-strength_beer_pac_per_capita',
    'cask_wine_pac_per_capita',
    'bottled_wine_pac_per_capita',
    'fortified_wine_pac_per_capita',
    'sin_month',
    'cos_month'
]

df['risk_class'] = pd.qcut(
    df['assault_rate'],
    q=3,
    labels=['low', 'medium', 'high']
)

X = df[feature_cols]
y = df['risk_class']

# ---------------------------------------------------------
# 3. TimeSeriesSplit
# ---------------------------------------------------------
tscv = TimeSeriesSplit(n_splits=5)

# ---------------------------------------------------------
# 4. Multinomial Logistic Regression CV
# ---------------------------------------------------------
logit_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        max_iter=500
    ))
])

logit_scores = cross_val_score(logit_pipe, X, y, cv=tscv)
print("Multinomial CV accuracy:", logit_scores.mean())

# ---------------------------------------------------------
# 5. Random Forest CV
# ---------------------------------------------------------
rf = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42
)

rf_scores = cross_val_score(rf, X, y, cv=tscv)
print("Random Forest CV accuracy:", rf_scores.mean())


c:\conda_envs\ai\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\conda_envs\ai\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\conda_envs\ai\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\conda_envs\ai\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7

Multinomial CV accuracy: nan
Random Forest CV accuracy: 0.4


In [11]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

# ---------------------------------------------------------
# 1. LOAD DATA (region × month)
# ---------------------------------------------------------
df = pd.read_csv("nt_crime_processed.csv")

# ---------------------------------------------------------
# 2. CREATE MONTHLY RISK CLASS
# ---------------------------------------------------------
df['risk_class'] = pd.qcut(
    df['assault_rate'],
    q=3,
    labels=['low', 'medium', 'high']
)


# ---------------------------------------------------------
# 4. SELECT FEATURES
# ---------------------------------------------------------
feature_cols = [
    'Total_population',
    'Alcohol involvement',
    'DV involvement',
    'pct_aboriginal',
    'pct_male',
    'pct_youth',
    'pct_senior',
    'alcohol_per_capita',
    'full-strength_beer_pac_per_capita',
    'mid-strength_beer_pac_per_capita',
    'low-strength_beer_pac_per_capita',
    'cask_wine_pac_per_capita',
    'bottled_wine_pac_per_capita',
    'fortified_wine_pac_per_capita',
    'sin_month',
    'cos_month'
]

# ---------------------------------------------------------
# 5. TRAIN ON 2024, TEST ON 2025
# ---------------------------------------------------------
train = df[df['Year'] == 2024]
test  = df[df['Year'] == 2025]

X_train = train[feature_cols]
y_train = train['risk_class']

X_test = test[feature_cols]
y_test = test['risk_class']

# ---------------------------------------------------------
# 6. CROSS-VALIDATION (2024 ONLY, TIME-AWARE)
# ---------------------------------------------------------
tscv = TimeSeriesSplit(n_splits=5)

# ---- Multinomial Logistic Regression CV ----
logit_pipe_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        max_iter=500
    ))
])

logit_cv_scores = cross_val_score(logit_pipe_cv, X_train, y_train, cv=tscv)
print("Multinomial CV accuracy (2024 only):", logit_cv_scores.mean())

# ---- Random Forest CV ----
rf_cv = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42
)

rf_cv_scores = cross_val_score(rf_cv, X_train, y_train, cv=tscv)
print("Random Forest CV accuracy (2024 only):", rf_cv_scores.mean())

# ---------------------------------------------------------
# 7. TRAIN FINAL MODELS ON 2024
# ---------------------------------------------------------

# ---- Multinomial Logistic Regression ----
logit_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        max_iter=500
    ))
])

logit_pipe.fit(X_train, y_train)
y_pred_logit = logit_pipe.predict(X_test)

print("\n=== MULTINOMIAL LOGISTIC REGRESSION (2025) ===")
print(confusion_matrix(y_test, y_pred_logit))
print(classification_report(y_test, y_pred_logit))

# ---- Random Forest ----
rf = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\n=== RANDOM FOREST (2025) ===")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# ---------------------------------------------------------
# 8. RANDOM FOREST FEATURE IMPORTANCE
# ---------------------------------------------------------
rf_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nRANDOM FOREST FEATURE IMPORTANCE")
print(rf_importance)


c:\conda_envs\ai\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\conda_envs\ai\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\conda_envs\ai\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\conda_envs\ai\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7

Multinomial CV accuracy (2024 only): nan
Random Forest CV accuracy (2024 only): 0.54

=== MULTINOMIAL LOGISTIC REGRESSION (2025) ===
[[15  1  2]
 [ 0 13  5]
 [ 9  6  9]]
              precision    recall  f1-score   support

        high       0.62      0.83      0.71        18
         low       0.65      0.72      0.68        18
      medium       0.56      0.38      0.45        24

    accuracy                           0.62        60
   macro avg       0.61      0.64      0.62        60
weighted avg       0.61      0.62      0.60        60



c:\conda_envs\ai\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



=== RANDOM FOREST (2025) ===
[[15  1  2]
 [ 0 15  3]
 [ 9  6  9]]
              precision    recall  f1-score   support

        high       0.62      0.83      0.71        18
         low       0.68      0.83      0.75        18
      medium       0.64      0.38      0.47        24

    accuracy                           0.65        60
   macro avg       0.65      0.68      0.65        60
weighted avg       0.65      0.65      0.63        60


RANDOM FOREST FEATURE IMPORTANCE
                              feature  importance
5                           pct_youth    0.164932
14                          sin_month    0.101417
1                 Alcohol involvement    0.083144
7                  alcohol_per_capita    0.079860
8   full-strength_beer_pac_per_capita    0.074375
9    mid-strength_beer_pac_per_capita    0.072610
10   low-strength_beer_pac_per_capita    0.070739
4                            pct_male    0.054552
11           cask_wine_pac_per_capita    0.050333
15                